In [21]:
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from medmnist.dataset import PneumoniaMNIST

## Image Transformation

In [22]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.ToTensor()
])

## Load Data

In [23]:
train_dataset = PneumoniaMNIST(
    split="train",
    transform=transform,
    download=True
)

val_dataset = PneumoniaMNIST(
    split="val",
    transform=transform,
    download=True
)

test_dataset = PneumoniaMNIST(
    split="test",
    transform=transform,
    download=True
)

In [24]:
train_dataset

Dataset PneumoniaMNIST of size 28 (pneumoniamnist)
    Number of datapoints: 4708
    Root location: C:\Users\Yash pal\.medmnist
    Split: train
    Task: binary-class
    Number of channels: 1
    Meaning of labels: {'0': 'normal', '1': 'pneumonia'}
    Number of samples: {'train': 4708, 'val': 524, 'test': 624}
    Description: The PneumoniaMNIST is based on a prior dataset of 5,856 pediatric chest X-Ray images. The task is binary-class classification of pneumonia against normal. We split the source training set with a ratio of 9:1 into training and validation set and use its source validation set as the test set. The source images are gray-scale, and their sizes are (384−2,916)×(127−2,713). We center-crop the images and resize them into 1×28×28.
    License: CC BY 4.0

## DataLoaders

In [25]:
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

In [26]:
img, labels = next(iter(train_loader))

print("Images shape:", img.shape)
print("Labels shape:", labels.shape)

Images shape: torch.Size([64, 1, 28, 28])
Labels shape: torch.Size([64, 1])


## Build the CNN

In [27]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            # First convolution block
            nn.Conv2d(
                in_channels=1,
                out_channels=16,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            # Second convolution block
            nn.Conv2d(
                in_channels=16,
                out_channels=32,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )

        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 64),
            nn.ReLU(),
            nn.Linear(64, 2),
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)

        return x

In [28]:
model = CNN()
print(model)

CNN(
  (conv_layers): Sequential(
    (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc_layers): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=1568, out_features=64, bias=True)
    (2): ReLU()
    (3): Linear(in_features=64, out_features=2, bias=True)
  )
)


In [29]:
images, labels = next(iter(train_loader))
outputs = model(images)

print("Input shape:", images.shape)
print("Output shape:", outputs.shape)

Input shape: torch.Size([64, 1, 28, 28])
Output shape: torch.Size([64, 2])


In [30]:
# Loss function

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

## Train the CNN

In [31]:
epochs = 20
train_losses = []

for epoch in range(epochs):
    model.train()
    epoch_training_loss = 0.0

    for images, labels in train_loader:
        optimizer.zero_grad()

        labels = labels.long().view(-1)

        output = model.forward(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()

        epoch_training_loss += loss.item()

    train_loss = epoch_training_loss / len(train_loader)
    train_losses.append(train_loss)

    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"Train Loss: {train_loss:.4f} "
    )

Epoch [1/20] Train Loss: 0.5193 
Epoch [2/20] Train Loss: 0.2237 
Epoch [3/20] Train Loss: 0.1510 
Epoch [4/20] Train Loss: 0.1480 
Epoch [5/20] Train Loss: 0.1286 
Epoch [6/20] Train Loss: 0.1241 
Epoch [7/20] Train Loss: 0.1295 
Epoch [8/20] Train Loss: 0.1120 
Epoch [9/20] Train Loss: 0.1050 
Epoch [10/20] Train Loss: 0.1001 
Epoch [11/20] Train Loss: 0.1055 
Epoch [12/20] Train Loss: 0.1077 
Epoch [13/20] Train Loss: 0.0953 
Epoch [14/20] Train Loss: 0.0924 
Epoch [15/20] Train Loss: 0.0901 
Epoch [16/20] Train Loss: 0.0854 
Epoch [17/20] Train Loss: 0.0795 
Epoch [18/20] Train Loss: 0.0787 
Epoch [19/20] Train Loss: 0.0768 
Epoch [20/20] Train Loss: 0.0707 


## Evaluation

In [32]:
correct_labels = 0
total_labels = 0
running_val_loss = 0.0
val_losses = []

model.eval()

with torch.no_grad():
    for images, labels in val_loader:
        labels = labels.long().view(-1)
        
        outputs = model.forward(images)
        
        loss = criterion(outputs, labels)
        running_val_loss += loss.item()
        
        _, predicted = torch.max(outputs, 1)


        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

val_loss = running_val_loss / len(val_loader)
val_losses.append(val_loss)
print(f"Val Loss: {val_loss:.4f}")
print(f"accuracy = {correct_labels / total_labels * 100}")

Val Loss: 0.1252
accuracy = 95.41984732824427
